In [1]:
import numpy as np
import pandas as pd

data = pd.read_csv('cleaned_fra_eng.csv')

In [2]:
data.head(20)

,English,French
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !
5,Hi.,Salut.
6,Run!,Cours !
7,Run!,Courez !
8,Run!,Prenez vos jambes à vos cous !
9,Run!,File !


In [3]:
data.shape

(240521, 2)

In [20]:
# Okay so our data has 2 columns --> 1st is the input sequence and 2nd is the output sequence. 
# Our first job would be to preprocess the data so that data becomes clean 

In [21]:
import pandas as pd
import re


for col in ['English', 'French']:
    # all data is treated as strings and convert to lowercase
    data[col] = data[col].astype(str).str.lower()
    
    # Remove symbols but keep letters and numbers.
    data[col] = data[col].str.replace(r'[^\w\s]', ' ', regex=True)
    
    # The \w in regex technically includes underscores. We'll strip those out explicitly.
    data[col] = data[col].str.replace('_', ' ', regex=False)
    
    # Fix the spacing: replace multiple consecutive spaces with a single space, 
    # and strip any leading/trailing spaces from the ends of the sentences.
    data[col] = data[col].str.replace(r'\s+', ' ', regex=True).str.strip()

# Check your cleaned data
print(data.head(10))

eng = data[['English']]
fra = data[['French']]

  English                        French
0      go                            va
1      go                        marche
2      go                      en route
3      go                         bouge
4      hi                         salut
5      hi                         salut
6     run                         cours
7     run                        courez
8     run  prenez vos jambes à vos cous
9     run                          file


In [22]:
import tiktoken
encoder = tiktoken.get_encoding("cl100k_base")

# Apply encoding to create new tokenized columns
data['English_Tokens'] = data['English'].apply(lambda x: encoder.encode(x))
data['French_Tokens'] = data['French'].apply(lambda x: encoder.encode(x))

# Count the number of tokens per row (highly useful for LLM data tracking)
data['English_Token_Count'] = data['English_Tokens'].apply(len)

print(data[['English', 'English_Tokens', 'English_Token_Count']].head(10))

  English English_Tokens  English_Token_Count
0      go         [3427]                    1
1      go         [3427]                    1
2      go         [3427]                    1
3      go         [3427]                    1
4      hi         [6151]                    1
5      hi         [6151]                    1
6     run         [6236]                    1
7     run         [6236]                    1
8     run         [6236]                    1
9     run         [6236]                    1


In [25]:
from collections import Counter

# Dictionaries to store mappings for both languages
word_to_id_dict = {}
id_to_word_dict = {}

special_tokens = ['[PAD]', '[UNK]', '[SOS]', '[EOS]']

for col in ['English', 'French']:
    # Extract and split all words for this specific language
    all_words = []
    for sentence in data[col]:
        all_words.extend(sentence.split())
        
    # Count frequencies
    word_counts = Counter(all_words)
    most_common_words = word_counts.most_common(10000)  # Keep top 10k words
    
    # Initialize mapping with special control tokens
    w2i = {token: idx for idx, token in enumerate(special_tokens)}
    
    # Append unique words to the mapping
    for word, count in most_common_words:
        w2i[word] = len(w2i)
        
    # Create the reverse lookup mapping
    i2w = {idx: word for word, idx in w2i.items()}
    
    # Store in global dictionaries
    word_to_id_dict[col] = w2i
    id_to_word_dict[col] = i2w

# Extract independent variables for clear usage
eng_w2i, eng_i2w = word_to_id_dict['English'], id_to_word_dict['English']
fra_w2i, fra_i2w = word_to_id_dict['French'], id_to_word_dict['French']

print(f"Unique English Word Vocabulary: {len(eng_w2i)}")
print(f"Unique French Word Vocabulary: {len(fra_w2i)}")


Unique English Word Vocabulary: 10004
Unique French Word Vocabulary: 10004


In [26]:
def encode_with_vocab(sentence, vocab_mapping):
    unk_id = vocab_mapping['[UNK]']
    # Wrap sentences with Start-of-Sentence and End-of-Sentence tokens for your model
    encoded = [vocab_mapping['[SOS]']] 
    encoded.extend([vocab_mapping.get(word, unk_id) for word in sentence.split()])
    encoded.append(vocab_mapping['[EOS]'])
    return encoded

# Apply language-specific encoders to respective columns
data['English_IDs'] = data['English'].apply(lambda x: encode_with_vocab(x, eng_w2i))
data['French_IDs'] = data['French'].apply(lambda x: encode_with_vocab(x, fra_w2i))

print("\nSample Translation Pair Vectorized:")
print(data[['English_IDs', 'French_IDs']].head(1))



Sample Translation Pair Vectorized:
  English_IDs   French_IDs
0  [2, 51, 3]  [2, 106, 3]


In [ ]:
# Now comes the main architecture part
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, rnn_type='GRU', num_layers=1, dropout=0.1):
        super(Encoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.rnn_type = rnn_type
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        if rnn_type == 'LSTM':
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, 
                               batch_first=True, dropout=dropout if num_layers > 1 else 0.0, bidirectional=True)
        else:
            self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, 
                              batch_first=True, dropout=dropout if num_layers > 1 else 0.0, bidirectional=True)
            
        # Since we use a bidirectional RNN, we project the combined forward/backward dimensions 
        # back down to a single hidden dimension size for the decoder setup
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_cell = nn.Linear(hidden_dim * 2, hidden_dim)

    def forward(self, source_tokens):
        # source_tokens shape: [batch_size, seq_len]
        embedded = self.embedding(source_tokens) # [batch_size, seq_len, embedding_dim]
        
        encoder_outputs, hidden = self.rnn(embedded)
        # encoder_outputs shape: [batch_size, seq_len, hidden_dim * 2]
        
        if self.rnn_type == 'LSTM':
            h_n, c_n = hidden
            h_concat = torch.cat((h_n[-2,:,:], h_n[-1,:,:]), dim=1)
            c_concat = torch.cat((c_n[-2,:,:], c_n[-1,:,:]), dim=1)
            
            decoder_hidden = torch.tanh(self.fc_hidden(h_concat)).unsqueeze(0).repeat(self.num_layers, 1, 1)
            decoder_cell = torch.tanh(self.fc_cell(c_concat)).unsqueeze(0).repeat(self.num_layers, 1, 1)
            return encoder_outputs, (decoder_hidden, decoder_cell)
        else:
            h_concat = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
            decoder_hidden = torch.tanh(self.fc_hidden(h_concat)).unsqueeze(0).repeat(self.num_layers, 1, 1)
            return encoder_outputs, decoder_hidden


class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(BahdanauAttention, self).__init__()
        # W1 scores the decoder's hidden state, W2 scores the encoder's outputs
        self.W1 = nn.Linear(hidden_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1)

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden shape: [num_layers, batch_size, hidden_dim] -> use top layer: [batch_size, hidden_dim]
        # encoder_outputs shape: [batch_size, seq_len, hidden_dim * 2]
        
        seq_len = encoder_outputs.shape[1]
        
        # Take the final layer's hidden state of the decoder
        dec_h = decoder_hidden[-1].unsqueeze(1) # [batch_size, 1, hidden_dim]
        dec_h = dec_h.repeat(1, seq_len, 1) # [batch_size, seq_len, hidden_dim]
        
        # Calculate Bahdanau Attention energy alignment score
        score = torch.tanh(self.W1(dec_h) + self.W2(encoder_outputs)) # [batch_size, seq_len, hidden_dim]
        attention_weights = F.softmax(self.V(score), dim=1) # [batch_size, seq_len, 1]
        
        # Multiply attention weights by encoder hidden states to get context vector
        context_vector = attention_weights * encoder_outputs # [batch_size, seq_len, hidden_dim * 2]
        context_vector = torch.sum(context_vector, dim=1) # [batch_size, hidden_dim * 2]
        
        return context_vector, attention_weights

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, attention, rnn_type='GRU', num_layers=1, dropout=0.1):
        super(Decoder, self).__init__()
        self.vocab_size = vocab_size
        self.rnn_type = rnn_type
        self.attention = attention
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # RNN input space combines the target word embedding and the encoder context vector
        rnn_input_dim = embedding_dim + (hidden_dim * 2)
        
        if rnn_type == 'LSTM':
            self.rnn = nn.LSTM(rnn_input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        else:
            self.rnn = nn.GRU(rnn_input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
            
        # Final fully connected layer outputs vocabulary probability distributions
        self.fc_out = nn.Linear(hidden_dim + (hidden_dim * 2) + embedding_dim, vocab_size)

    def forward(self, target_token, decoder_hidden, encoder_outputs):
        target_token = target_token.unsqueeze(1) # [batch_size, 1]
        
        embedded = self.embedding(target_token) # [batch_size, 1, embedding_dim]
        
        current_h = decoder_hidden[0] if self.rnn_type == 'LSTM' else decoder_hidden
        context, attention_weights = self.attention(current_h, encoder_outputs) # [batch_size, hidden_dim * 2]
        
        context_input = context.unsqueeze(1) 
        rnn_input = torch.cat((embedded, context_input), dim=2) 
        
        rnn_output, decoder_hidden = self.rnn(rnn_input, decoder_hidden)
        
        prediction_input = torch.cat((rnn_output.squeeze(1), context, embedded.squeeze(1)), dim=1)
        predictions = self.fc_out(prediction_input)
        
        return predictions, decoder_hidden, attention_weights


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        
    def forward(self, source, target, teacher_forcing_ratio=0.5):
        batch_size = source.shape[0]
        target_len = target.shape[1]
        target_vocab_size = self.decoder.vocab_size
        
        # Tensor to store decoder token output predictions
        outputs = torch.zeros(batch_size, target_len, target_vocab_size).to(source.device)
        
        # Run source sequence through encoder
        encoder_outputs, decoder_hidden = self.encoder(source)
        
        # First input token to the decoder is always the Start-of-Sentence ([SOS]) token
        input_token = target[:, 0]
        
        for t in range(1, target_len):
            output, decoder_hidden, _ = self.decoder(input_token, decoder_hidden, encoder_outputs)
            outputs[:, t, :] = output
            
            # Decide whether to use actual ground truth or the model's prediction for the next step input
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1) 
            input_token = target[:, t] if teacher_force else top1
            
        return outputs


In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import optuna
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def prepare_real_data(df, eng_w2i, fra_w2i):

    def encode_sentence(sentence, vocab_mapping):
        unk_id = vocab_mapping['[UNK]']
        # Build boundary tokens: [SOS] + structural subwords + [EOS]
        encoded = [vocab_mapping['[SOS]']] 
        encoded.extend([vocab_mapping.get(word, unk_id) for word in str(sentence).split()])
        encoded.append(vocab_mapping['[EOS]'])
        return encoded

    # Vectorize columns dynamically
    encoded_eng = df['English'].apply(lambda x: encode_sentence(x, eng_w2i)).tolist()
    encoded_fra = df['French'].apply(lambda x: encode_sentence(x, fra_w2i)).tolist()
    
    return encoded_eng, encoded_fra


class TranslationDataset(Dataset):
    def __init__(self, src_sequences, trg_sequences):
        self.src_sequences = [torch.tensor(seq, dtype=torch.long) for seq in src_sequences]
        self.trg_sequences = [torch.tensor(seq, dtype=torch.long) for seq in trg_sequences]

    def __len__(self):
        return len(self.src_sequences)

    def __getitem__(self, idx):
        return self.src_sequences[idx], self.trg_sequences[idx]

def pad_collate_fn(batch):
 
    src_batch, trg_batch = zip(*batch)
    src_padded = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0) # [PAD] ID = 0
    trg_padded = nn.utils.rnn.pad_sequence(trg_batch, batch_first=True, padding_value=0)
    return src_padded, trg_padded

def translate_sentence(model, source_tensor, max_len=50, sos_token=2, eos_token=3):

    model.eval()
    with torch.no_grad():
        encoder_outputs, decoder_hidden = model.encoder(source_tensor.unsqueeze(0).to(device))
        translated_tokens = [sos_token]
        
        for _ in range(max_len):
            current_token = torch.tensor([translated_tokens[-1]], dtype=torch.long).to(device)
            output, decoder_hidden, _ = model.decoder(current_token, decoder_hidden, encoder_outputs)
            
            # Select word with the absolute highest activation mapping
            predicted_token = output.argmax(1).item()
            translated_tokens.append(predicted_token)
            
            if predicted_token == eos_token:
                break
                
    return translated_tokens

def evaluate_bleu(model, dataset, sos_token=2, eos_token=3):
    model.eval()
    bleu_scores = []
    smooth_fn = SmoothingFunction().method1
    
    for idx in range(min(len(dataset), 150)): 
        src_tensor, trg_tensor = dataset[idx]
        
        reference = [tok for tok in trg_tensor.tolist() if tok not in [0, sos_token, eos_token]]
        generated_seq = translate_sentence(model, src_tensor, sos_token=sos_token, eos_token=eos_token)
        hypothesis = [tok for tok in generated_seq if tok not in [0, sos_token, eos_token]]
        
        if not reference or not hypothesis:
            continue
            
        score = sentence_bleu([reference], hypothesis, smoothing_function=smooth_fn)
        bleu_scores.append(score)
        
    return sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0


def objective(trial, train_dataset, val_dataset, src_vocab_size, trg_vocab_size):
    embedding_dim = trial.suggest_int('embedding_dim', 64, 256, step=64)
    hidden_dim = trial.suggest_int('hidden_dim', 128, 512, step=128)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    rnn_type = trial.suggest_categorical('rnn_type', ['GRU', 'LSTM'])
    dropout = trial.suggest_float('dropout', 0.1, 0.4)
    teacher_forcing_ratio = trial.suggest_float('teacher_forcing_ratio', 0.3, 0.7)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                              collate_fn=pad_collate_fn, num_workers=0, pin_memory=True)
    
    encoder = Encoder(vocab_size=src_vocab_size, embedding_dim=embedding_dim, 
                      hidden_dim=hidden_dim, rnn_type=rnn_type, num_layers=1, dropout=dropout)
    attention = BahdanauAttention(hidden_dim=hidden_dim)
    decoder = Decoder(vocab_size=trg_vocab_size, embedding_dim=embedding_dim, 
                      hidden_dim=hidden_dim, attention=attention, rnn_type=rnn_type, num_layers=1, dropout=dropout)
    
    model = Seq2Seq(encoder, decoder).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # LOSS CONFIG: Exclude special tracking index spaces [PAD]=0, [SOS]=2, [EOS]=3
    weight_mask = torch.ones(trg_vocab_size).to(device)
    weight_mask[0] = 0.0  
    weight_mask[2] = 0.0  
    weight_mask[3] = 0.0  
    
    criterion = nn.CrossEntropyLoss(weight=weight_mask)
    
    epochs = 3
    for epoch in range(epochs):
        model.train()
        for src, trg in train_loader:
            src, trg = src.to(device), trg.to(device)
            optimizer.zero_grad(set_to_none=True)
            
            output = model(src, trg, teacher_forcing_ratio=teacher_forcing_ratio)
            
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)
            
            loss = criterion(output, trg)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
    val_bleu = evaluate_bleu(model, val_dataset)
    return val_bleu


if __name__ == "__main__":
    # eng_tokens, fra_tokens = prepare_real_data(data, eng_w2i, fra_w2i)
    
    mock_eng_tokens = [[2, 14, 55, 23, 3], [2, 9, 88, 3]] * 250
    mock_fra_tokens = [[2, 41, 19, 82, 3], [2, 7, 34, 3]] * 250
    
    dataset = TranslationDataset(mock_eng_tokens, mock_fra_tokens)
    
    # Split into separate partitions to isolate training performance
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_data, val_data = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    ENGLISH_VOCAB_SIZE = 5000
    FRENCH_VOCAB_SIZE = 5000

    db_file = "optuna_study.db"
    storage_url = f"sqlite:///{db_file}"
    
    study = optuna.create_study(
        study_name="real_data_translation_tuning",
        storage=storage_url,
        direction="maximize",
        pruner=optuna.pruners.MedianPruner(),
        load_if_exists=True
    )

    print("Running parallel tuning loops using actual dataset streams...")
    study.optimize(lambda trial: objective(trial, train_data, val_data, ENGLISH_VOCAB_SIZE, FRENCH_VOCAB_SIZE), 
                   n_trials=5, n_jobs=-1)

    print("\nTuning Complete. Optimal Configuration Framework Saved:")
    print(study.best_trial.params)
    
    with open("best_hyperparameters.json", "w") as f:
        json.dump(study.best_trial.params, f, indent=4)


[I 2026-08-14 14:33:06,415] Using an existing study with name 'real_data_translation_tuning' instead of creating a new one.


Running parallel tuning loops using actual dataset streams...


[I 2026-08-14 14:33:24,973] Trial 5 finished with value: 0.012641802874862162 and parameters: {'embedding_dim': 256, 'hidden_dim': 128, 'learning_rate': 0.002206241412973771, 'batch_size': 64, 'rnn_type': 'LSTM', 'dropout': 0.21764557896994818, 'teacher_forcing_ratio': 0.32241036700857106}. Best is trial 0 with value: 0.013161581670421476.
[I 2026-08-14 14:33:25,762] Trial 4 finished with value: 0.012641802874862162 and parameters: {'embedding_dim': 256, 'hidden_dim': 256, 'learning_rate': 0.001432813655208919, 'batch_size': 64, 'rnn_type': 'GRU', 'dropout': 0.18197850144611638, 'teacher_forcing_ratio': 0.6172607274336996}. Best is trial 0 with value: 0.013161581670421476.
[I 2026-08-14 14:33:26,878] Trial 3 finished with value: 0.012641802874862162 and parameters: {'embedding_dim': 192, 'hidden_dim': 384, 'learning_rate': 0.001945188705727849, 'batch_size': 32, 'rnn_type': 'LSTM', 'dropout': 0.11296070331629841, 'teacher_forcing_ratio': 0.3846679462483076}. Best is trial 0 with value:


Tuning Complete. Optimal Configuration Framework Saved:
{'embedding_dim': 192, 'hidden_dim': 384, 'learning_rate': 0.0005845770509965827, 'batch_size': 64, 'rnn_type': 'LSTM', 'dropout': 0.28166034336441936, 'teacher_forcing_ratio': 0.6960344527207363}


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import json
import time

with open("best_hyperparameters.json", "r") as f:
    best_config = json.load(f)

print(f"Training with optimal parameters: {best_config}")

ENG_VOCAB_SIZE = len(eng_w2i)
FRA_VOCAB_SIZE = len(fra_w2i)

print(f"Using dynamic vocab sizes - English: {ENG_VOCAB_SIZE}, French: {FRA_VOCAB_SIZE}")

encoded_eng, encoded_fra = prepare_real_data(data, eng_w2i, fra_w2i)

# Create the dataset and dataloader
full_dataset = TranslationDataset(encoded_eng, encoded_fra)
train_loader = DataLoader(
    full_dataset, 
    batch_size=best_config['batch_size'], 
    shuffle=True, 
    collate_fn=pad_collate_fn
)

# INITIALIZE ARCHITECTURE
encoder = Encoder(
    vocab_size=ENG_VOCAB_SIZE, 
    embedding_dim=best_config['embedding_dim'], 
    hidden_dim=best_config['hidden_dim'], 
    rnn_type=best_config['rnn_type'], 
    num_layers=1, 
    dropout=best_config['dropout']
)

attention = BahdanauAttention(hidden_dim=best_config['hidden_dim'])

decoder = Decoder(
    vocab_size=FRA_VOCAB_SIZE, 
    embedding_dim=best_config['embedding_dim'], 
    hidden_dim=best_config['hidden_dim'], 
    attention=attention, 
    rnn_type=best_config['rnn_type'], 
    num_layers=1, 
    dropout=best_config['dropout']
)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Seq2Seq(encoder, decoder).to(device)

# OPTIMIZER & LOSS FUNCTION
optimizer = optim.Adam(model.parameters(), lr=best_config['learning_rate'])

# We MUST ignore the padding token (Index 0) so the model isn't penalized for empty space
criterion = nn.CrossEntropyLoss(ignore_index=0)

EPOCHS = 5 
teacher_forcing_ratio = best_config['teacher_forcing_ratio']

print(f"\nStarting training on {device} for {EPOCHS} epochs")

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    start_time = time.time()
    
    for batch_idx, (src, trg) in enumerate(train_loader):
        src, trg = src.to(device), trg.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        
        # Forward pass
        output = model(src, trg, teacher_forcing_ratio)
        
        # Reshape for loss function
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)
        
        # Calculate loss and backpropagate
        loss = criterion(output, trg)
        loss.backward()
        
        # Clip gradients to prevent exploding gradients in RNNs
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # Print progress every 500 batches so your notebook doesn't get flooded
        if batch_idx % 500 == 0:
            print(f"Epoch: {epoch+1}/{EPOCHS} | Batch: {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    end_time = time.time()
    avg_loss = epoch_loss / len(train_loader)
    print(f"---> Epoch {epoch+1} Complete | Avg Loss: {avg_loss:.4f} | Time: {end_time - start_time:.2f}s \n")

# SAVE THE FINAL MODEL
torch.save(model.state_dict(), "best_translation_model.pt")
print("Training complete, Model saved as 'best_translation_model.pt'.")

Training with optimal parameters: {'embedding_dim': 192, 'hidden_dim': 384, 'learning_rate': 0.0005845770509965827, 'batch_size': 64, 'rnn_type': 'LSTM', 'dropout': 0.28166034336441936, 'teacher_forcing_ratio': 0.6960344527207363}
Using dynamic vocab sizes - English: 10004, French: 10004

Starting training on cuda for 5 epochs...
Epoch: 1/5 | Batch: 0/3759 | Loss: 9.2334
Epoch: 1/5 | Batch: 500/3759 | Loss: 2.9564
Epoch: 1/5 | Batch: 1000/3759 | Loss: 2.6449
Epoch: 1/5 | Batch: 1500/3759 | Loss: 2.0499
Epoch: 1/5 | Batch: 2000/3759 | Loss: 2.3256
Epoch: 1/5 | Batch: 2500/3759 | Loss: 1.8125
Epoch: 1/5 | Batch: 3000/3759 | Loss: 1.6722
Epoch: 1/5 | Batch: 3500/3759 | Loss: 1.9796
--- Epoch 1 Complete | Avg Loss: 2.3635 | Time: 568.19s ---

Epoch: 2/5 | Batch: 0/3759 | Loss: 1.2813
Epoch: 2/5 | Batch: 500/3759 | Loss: 1.3889
Epoch: 2/5 | Batch: 1000/3759 | Loss: 2.0254
Epoch: 2/5 | Batch: 1500/3759 | Loss: 1.4499
Epoch: 2/5 | Batch: 2000/3759 | Loss: 1.1540
Epoch: 2/5 | Batch: 2500/3759 

In [ ]:
import json
import re
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open("best_hyperparameters.json", "r") as f:
    best_config = json.load(f)

print("Loaded Optimal Configuration:", best_config)

ENGLISH_VOCAB_SIZE = len(eng_w2i)
FRENCH_VOCAB_SIZE = len(fra_i2w)

encoder = Encoder(
    vocab_size=ENGLISH_VOCAB_SIZE, 
    embedding_dim=best_config['embedding_dim'], 
    hidden_dim=best_config['hidden_dim'], 
    rnn_type=best_config['rnn_type'], 
    num_layers=1, 
    dropout=best_config['dropout']
)

attention = BahdanauAttention(hidden_dim=best_config['hidden_dim'])

decoder = Decoder(
    vocab_size=FRENCH_VOCAB_SIZE, 
    embedding_dim=best_config['embedding_dim'], 
    hidden_dim=best_config['hidden_dim'], 
    attention=attention, 
    rnn_type=best_config['rnn_type'], 
    num_layers=1, 
    dropout=best_config['dropout']
)

# Initialize wrapper and load your trained weights
model = Seq2Seq(encoder, decoder).to(device)
model.load_state_dict(torch.load("best_translation_model.pt", map_location=device)) # Uncomment when model file is saved
model.eval()



# TEXT CLEANING & REAL-TIME INFERENCE
def clean_input_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = text.replace('_', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def translate_user_input(sentence, eng_w2i, fra_i2w):
    # Standardize string structure
    cleaned_text = clean_input_text(sentence)
    
    # Numerical translation encoding with [SOS] and [EOS] tokens
    unk_id = eng_w2i.get('[UNK]', 1)
    tokens = [eng_w2i.get('[SOS]', 2)]
    tokens.extend([eng_w2i.get(word, unk_id) for word in cleaned_text.split()])
    tokens.append(eng_w2i.get('[EOS]', 3))
    
    source_tensor = torch.tensor(tokens, dtype=torch.long).to(device)
    
    # Generate prediction tokens via the trained model autoregressively
    predicted_ids = translate_sentence(model, source_tensor, max_len=50)
    
    # Convert predicted IDs back to readable word
    output_words = []
    for idx in predicted_ids:
        if idx in [0,2]:
            continue
        if idx == 3: # Break on [EOS]
            break
        output_words.append(fra_i2w.get(idx, '[UNK]'))
        
    return " ".join(output_words)


# INTERACTIVE EXECUTION LOOP
if __name__ == "__main__":

    print("\n--- Translation ---")
    user_sentence = "Hello_World!"
    
    translation = translate_user_input(user_sentence, eng_w2i, fra_i2w)
    print(f"Input Sentence:  {user_sentence}")
    print(f"Cleaned Vector:  {clean_input_text(user_sentence)}")
    print(f"Model Output:   {translation}")


Loaded Optimal Configuration: {'embedding_dim': 192, 'hidden_dim': 384, 'learning_rate': 0.0005845770509965827, 'batch_size': 64, 'rnn_type': 'LSTM', 'dropout': 0.28166034336441936, 'teacher_forcing_ratio': 0.6960344527207363}

--- Translation Service Active ---
Input Sentence:  Hello_World!
Cleaned Vector:  hello world
Model Output:   salut le monde


In [ ]:
import json

with open("eng_w2i.json", "w") as f:
    json.dump(eng_w2i, f)

with open("fra_i2w.json", "w") as f:
    json.dump(fra_i2w, f)
    
print("Vocabularies saved successfully!")

Vocabularies saved successfully!
